# 挑战：

- 构建一个综合数据集生成器，编写可以生成数据集的模型。
- 使用多种模型和提示进行不同的输出，尝试量化和不量化，尝试不同的形状和大小。
- 为您的产品创建 Gradio UI

## 想法

- 生成开发 REST API 时使用的示例数据。

## ⚠️ 在运行此笔记本之前

**需要 GPU。** 此笔记本电脑通过 BitsAndBytes 使用 4 位量化模型，并需要支持 CUDA 的 NVIDIA GPU。不支持 CPU 推理。

**磁盘空间。** 模型在首次运行时从 Hugging Face 下载并缓存在本地 (`~/.cache/huggingface/hub`)。下载是每个模型的一次性费用：

|型号|下载大小（大约）|
|--------|----------------------|
|骆驼 3.1 8B 指导 | 〜16 GB |
| Qwen 2.5 Coder 7B 指导 | 〜15 GB |
| Phi 4 迷你指导 | 〜5 GB |

**总计：首次运行时约为 36 GB**。后续运行从缓存加载。

**HF 代币。** Llama 需要一个 Hugging Face 帐户，其访问权限已在 [meta-llama/Meta-Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct) 上获得批准。在“.env”文件中设置“HF_TOKEN”。

---

## 安装顺序很重要

首先安装支持 CUDA 的 PyTorch，然后使用“--no-deps”安装其他软件包，以防止它们引入仅 CPU 的 torch 构建：```bash
uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
uv pip install --no-deps accelerate bitsandbytes transformers==4.57.6 huggingface_hub[hf_xet]
```在本地环境中，这些安装是持久的 - 您只需运行它们一次。

In [ ]:
import torch, accelerate, bitsandbytes, transformers

print(f"torch=={torch.__version__}")
print(f"accelerate=={accelerate.__version__}")
print(f"bitsandbytes=={bitsandbytes.__version__}")
print(f"transformers=={transformers.__version__}")

In [ ]:
# 进口

from transformers import (
    pipeline,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
import os
from dotenv import load_dotenv
from openai import OpenAI
from huggingface_hub import login

load_dotenv(override=True)

In [ ]:
# 常数

LLAMA_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
QWEN_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
PHI_MODEL = "microsoft/Phi-4-mini-instruct"
OPENAPI_FILE = "storefront-sample.json"

In [ ]:
# 拥抱登录

login(token=os.getenv("HF_TOKEN"), add_to_git_credential=True)

In [ ]:
# 如果模式中使用 $ref -resolve_ref


def resolve_ref(spec, ref):
    path = ref.replace("#/", "").split("/")
    obj = spec
    for p in path:
        obj = obj[p]
    return obj

In [ ]:
# 提取功能

import json


def extract_schema(openapi, path, method):
    method = method.lower()
    endpoint = openapi["paths"][path][method]
    schema = endpoint["requestBody"]["content"]["application/json"]["schema"]
    return schema


# 注意：在此阶段，我们有意仅提取一个端点。
# 提取助手适用于任何路径/方法；单端点选择使笔记本电脑保持最小化。
ENDPOINT_PATH = "/cart/items"
ENDPOINT_METHOD = "post"

with open(OPENAPI_FILE) as f:
    spec = json.load(f)

schema = extract_schema(spec, ENDPOINT_PATH, ENDPOINT_METHOD)

if "$ref" in schema:
    schema = resolve_ref(spec, schema["$ref"])

print(json.dumps(schema, indent=2))

In [ ]:
# 有意保持尽可能少

user_prompt = f"""Generate realistic JSON objects following this schema:


{schema}
"""


system_prompt = """


You excel at generating realistic JSON objects following a given schema.
"""


messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [ ]:
import torch

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)


def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name, device_map="auto", quantization_config=quant_config
    )
    return model, tokenizer


def generate_json(model, tokenizer, messages, max_new_tokens=1000):
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", return_dict=True, add_generation_prompt=True
    ).to("cuda")

    print(f"  Input tokens: {inputs['input_ids'].shape[1]}")
    print(f"  Chat template found: {tokenizer.chat_template is not None}")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[1] :]
    print(f"  Output tokens: {len(new_tokens)}")
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
import re
from jsonschema import validate, ValidationError


def extract_json(text):
    """从模型输出中提取第一个完整的 JSON 对象或数组。"""
    if text is None:
        return None

    # 尝试```json ... ```首先是代码块
    code_block = re.search(r"```(?:json)?\s*(\{[\s\S]*?\}|\[[\s\S]*?\])\s*```", text)
    if code_block:
        return code_block.group(1).strip()

    # 回退到匹配最外面的大括号/方括号
    for start, ch in enumerate(text):
        if ch not in "{[":
            continue
        close = "}" if ch == "{" else "]"
        depth = 0
        for end in range(start, len(text)):
            if text[end] == ch:
                depth += 1
            elif text[end] == close:
                depth -= 1
                if depth == 0:
                    return text[start : end + 1]
    return None


def validate_json(text):
    """解析 JSON 字符串，返回 (parsed_object, is_valid) 元组。"""
    try:
        return json.loads(text), True
    except (json.JSONDecodeError, TypeError):
        return None, False


def validate_schema(data, target_schema):
    """检查数据是否符合目标模式。返回（符合，错误消息）。"""
    items = data if isinstance(data, list) else [data]
    errors = []
    for i, item in enumerate(items):
        try:
            validate(instance=item, schema=target_schema)
        except ValidationError as e:
            errors.append(f"Item {i}: {e.message}")
    if errors:
        return False, "; ".join(errors)
    return True, None

In [ ]:
import gc

MODELS = {
    "Llama 3.1 8B": LLAMA_MODEL,
    "Qwen 2.5 Coder 7B": QWEN_MODEL,
    "Phi 4 Mini": PHI_MODEL,
}

results = []

for label, model_name in MODELS.items():
    print(f"\n{'=' * 60}")
    print(f"  {label} ({model_name})")
    print(f"{'=' * 60}")

    try:
        model, tokenizer = load_model(model_name)
        raw_output = generate_json(model, tokenizer, messages)

        extracted = extract_json(raw_output)
        parsed, is_valid = validate_json(extracted)
        conforms, schema_errors = (
            validate_schema(parsed, schema) if is_valid else (False, "No valid JSON")
        )

        results.append(
            {
                "model": label,
                "valid_json": is_valid,
                "schema_match": conforms,
                "schema_errors": schema_errors,
                "raw_output": raw_output,
                "extracted": extracted,
                "parsed": parsed,
            }
        )

        print(f"\nRaw output:\n{raw_output}")
        print(f"\nExtracted JSON:\n{extracted}")
        print(f"\nValid JSON: {is_valid}")
        print(f"Schema match: {conforms}")
        if schema_errors:
            print(f"Schema errors: {schema_errors}")

        del model, tokenizer
        torch.cuda.empty_cache()
        gc.collect()

    except Exception as e:
        print(f"\nError: {e}")
        results.append(
            {
                "model": label,
                "valid_json": False,
                "raw_output": None,
                "extracted": None,
                "parsed": None,
                "error": str(e),
                "schema_match": False,
                "schema_errors": str(e),
            }
        )

In [ ]:
print(f"\n{'=' * 60}")
print("  Model Comparison Summary")
print(f"{'=' * 60}\n")
print(f"  {'Model':<25} {'Valid JSON':<14} {'Schema Match'}")
print(f"  {'-' * 53}")
for r in results:
    print(f"  {r['model']:<25} {str(r['valid_json']):<14} {r['schema_match']}")
    if r.get("schema_errors"):
        print(f"    -> {r['schema_errors']}")